<img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:100px" alt="MCUNet Logo">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Déploiement TinyML avec MCUNet </h1></center>
<center><h2> Notebook 00 : Vérification de l'environnement (Solution) </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

>
> **Objectif :** Vérifier que l'environnement Docker fonctionne correctement et que toutes les ressources nécessaires (données, modèles, bibliothèques) sont disponibles.
>
> **Durée estimée :** 10 minutes
>
> **Prérequis :** Avoir démarré le container Docker avec `make start`.
>


### Concepts clés
>
> 1. **TinyML** : Exécution de modèles de Machine Learning sur des ressources extrêmement contraintes (microcontrôleurs).
> 2. **MCUNet** : Un framework co-conçu (système + algorithme) par le MIT pour le TinyML.
> 3. **Environnement figé** : Le TinyML repose souvent sur des versions très spécifiques de bibliothèques (ici TensorFlow 1.15) pour la compatibilité avec les outils d'exportation vers C/C++.


--- 
### 1. Vérification des bibliothèques Python

Nous devons nous assurer que les versions spécifiques demandées par MCUNet sont bien installées.

In [ ]:
import sys
import tensorflow as tf
import torch
import torchvision
import numpy as np
import tflite_runtime
import mcunet

print(f"Python version: {sys.version.split()[0]}")
print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"TFLite Runtime version: {tflite_runtime.__version__}")
print(f"MCUNet successfully imported!")

<div class="alert alert-warning">
<i class='fa fa-exclamation-triangle '></i> &emsp; 
  <strong>Attention :</strong> TensorFlow 1.15 affiche peut-être des warnings de dépréciation. C'est normal et attendu. C'est la version requise pour l'export TFLite stable vers certains outils MCU.
</div>

--- 
### 2. Vérification des modèles pré-téléchargés

Le Dockerfile a normalement pré-téléchargé plusieurs modèles pour nous faire gagner du temps.

In [ ]:
from mcunet.model_zoo import net_id_list
import os

print("Modèles disponibles dans MCUNet Model Zoo :")
for net_id in net_id_list:
    print(f" - {net_id}")

print("\nVérification du cache local des modèles TFLite...")
cache_dir = os.path.expanduser("~/.mcunet/") # Le dossier par defaut est ~/.mcunet/
if os.path.exists(cache_dir):
    files = os.listdir(cache_dir)
    tflite_files = [f for f in files if f.endswith('.tflite')]
    print(f"Trouvé {len(tflite_files)} modèles TFLite dans le cache :")
    for f in tflite_files:
        size_kb = os.path.getsize(os.path.join(cache_dir, f)) / 1024
        print(f" - {f} ({size_kb:.1f} KB)")
else:
    print("Dossier de cache non trouvé. Les modèles seront téléchargés à la volée.")

--- 
### 3. Vérification du Dataset VWW (Visual Wake Words)

VWW est la tâche "Hello World" de la vision sur microcontrôleur. L'objectif est de détecter si une personne est présente ou non sur une petite image.

In [ ]:
data_dir = "/dataset/vww_minival"

if os.path.exists(data_dir):
    # Compter les images
    person_dir = os.path.join(data_dir, "person")
    non_person_dir = os.path.join(data_dir, "non_person")
    
    n_person = len(os.listdir(person_dir)) if os.path.exists(person_dir) else 0
    n_non_person = len(os.listdir(non_person_dir)) if os.path.exists(non_person_dir) else 0
    
    print("Dataset VWW minival trouvé !")
    print(f"Images 'Person' : {n_person}")
    print(f"Images 'Non-Person' : {n_non_person}")
    print(f"Total : {n_person + n_non_person} images")
else:
    print("ERREUR : Dataset non trouvé. Vérifiez le processus de build Docker.")

--- 
### 4. Petit exercice pratique

Compléter la cellule ci-dessous pour charger une image aléatoire de la classe "person" et l'afficher avec la bibliothèque `matplotlib.pyplot` et `PIL.Image`.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# Solution
person_dir = os.path.join(data_dir, "person")
files = os.listdir(person_dir)
random_file = random.choice(files)

img_path = os.path.join(person_dir, random_file)
img = Image.open(img_path)

plt.imshow(img)
plt.title(f"Exemple de Visual Wake Words (Personne)\n{random_file}")
plt.axis('off')
plt.show()


--- 
### Questions de réflexion

1. **Taille du modèle vs Dataset** : Les modèles TinyML font souvent moins de 500 KB, pourtant notre dataset d'évaluation (minival) fait 380 MB. Pourquoi le dataset d'évaluation est-il si volumineux par rapport au modèle final ?

--> Le modèle a été entraîné sur un grand nombre de données, mais on n'en retire qu'une longue liste de paramètres, qui pèse bien moins lourd en mémoire. En plus de cela, on peut quantifier les poids, réduisant encore plus sa taille sur disque.

2. **Compatibilité ascendante** : Pourquoi est-il si critique d'utiliser TensorFlow 1.15 et non pas une version récente (comme TF 2.15) pour ce projet précis d'export vers microcontrôleur ?

--> MCUNet a été développé en 2019-2020, à l'époque où TF 1.x était le standard. Le code de conversion TFLite int8 a une API et un comportement différents entre TF1 et TF2 :

* L'API a changé (graphes statiques vs eager execution)
* Les opérateurs supportés en quantification ne sont pas strictement identiques
* Le format binaire des modèles TFLite int8 produits diffère subtilement (ordre des tenseurs, gestion des biais)

<div class="alert alert-success">
<i class="fa fa-check-circle"></i> &emsp; 
<strong>Setup OK à ce niveau.</strong> Si tout s'est affiché correctement et que l'image apparaît, votre environnement est prêt. Vous pouvez passer au notebook 01 !
</div>